# Demo Chinh Thuc Person Re-Identification Sau Khi Da Co Checkpoint

Notebook nay duoc thiet ke cho buoi demo chinh thuc sau khi hai mo hinh da train xong va da co checkpoint hop le.

Hai huong duoc demo trong cung mot notebook:
- **Local Reliability**
- **Semantic Reliability**

Muc tieu cua notebook:
- kiem tra workspace, dataset, semantic mask, config va checkpoint truoc khi chay;
- minh hoa ban chat bai toan Person ReID theo kieu query-gallery retrieval;
- chay evaluation chinh thuc bang checkpoint da train xong;
- hien thi cac chi so `mAP`, `Rank-1`, `Rank-5`, `Rank-10`;
- visualize top-k retrieval de thay truong hop dung va sai;
- visualize patch reliability, local weights va semantic mask de giai thich y tuong mo hinh.

Luu y:
- Notebook nay **yeu cau checkpoint hop le** cho cac approach duoc demo.
- Neu thieu checkpoint, notebook se dung som ngay o buoc preflight check de tranh evaluation loi ve sau.


In [ ]:
import os
import re
import sys
import math
import json
import shlex
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display

# Danh sach package noi bo cua hai repo. Khi chuyen qua repo khac, ta xoa cache import
# de tranh bi dung nham module cua repo con lai.
PACKAGE_PREFIXES = {'config', 'datasets', 'model', 'loss', 'processor', 'solver', 'utils'}

def find_workspace_root(start=None):
    # Tim workspace goc chua dong thoi hai repo local-reliability va semantic.
    current = Path.cwd().resolve() if start is None else Path(start).resolve()
    for candidate in [current] + list(current.parents):
        if (candidate / 'local-reliability').exists() and (candidate / 'semantic').exists():
            return candidate
    raise FileNotFoundError('Khong tim thay workspace root chua hai repo local-reliability va semantic.')

WORKSPACE = find_workspace_root()
REPOS = {
    'local_reliability': WORKSPACE / 'local-reliability',
    'semantic': WORKSPACE / 'semantic',
}
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
os.chdir(WORKSPACE)

# Cac bien duoi day la diem chinh can sua truoc khi bam Run All tren Colab.
DATA_ROOT = Path('/content/data')
DATASET = 'market1501'
DATASET_ROOT = DATA_ROOT / DATASET
PRETRAIN = Path('/content/drive/MyDrive/TGMT/PROJECT/Colab/req23/checkpoint/jx_vit_base_p16_224-80ecf9dd.pth')
TOPK = 10
QUERY_INDEX = 0
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

APPROACH = {
    'local_reliability': {
        'display_name': 'Local Reliability',
        'config': 'configs/Market/vit_transreid_stride_reliability_full_imagenet.yml',
        'output_dir': '/content/drive/MyDrive/TGMT/PROJECT/Colab/official_demo/local_reliability',
        'test_weight': None,
        'ims': 16,
        'workers': 2,
        'epochs': 5,
        'eval_period': 1,
        'ckpt_period': 1,
    },
    'semantic': {
        'display_name': 'Semantic Reliability',
        'config': 'configs/Market/vit_transreid_stride_sem_decoder_reliability.yml',
        'output_dir': '/content/drive/MyDrive/TGMT/PROJECT/Colab/official_demo/semantic',
        'test_weight': None,
        'ims': 16,
        'workers': 2,
        'epochs': 5,
        'eval_period': 1,
        'ckpt_period': 1,
        'mask_dir': DATASET_ROOT / 'semantic_groups',
        'raw_mask_dir': DATASET_ROOT / 'semantic_raw',
    },
}

print('WORKSPACE =', WORKSPACE)
print('DEVICE =', DEVICE)
print('DATASET_ROOT =', DATASET_ROOT)
for kind, spec in APPROACH.items():
    print(f"{kind}: config={spec['config']} | test_weight={spec['test_weight']}")


## Preflight Check

Section nay la buoc bat buoc truoc khi demo. Muc tieu la xac nhan workspace, dataset, semantic mask, pretrained backbone va checkpoint evaluation deu ton tai.

Voi notebook demo chinh thuc, neu checkpoint chua duoc gan vao `APPROACH[kind]['test_weight']` hoac file khong ton tai, notebook se dung som bang `FileNotFoundError` de tranh chay den cac cell evaluation moi bi loi.


In [ ]:
IMG_EXTS = {'.jpg', '.jpeg', '.png', '.bmp'}
PAL = np.array([
    [0, 0, 0],
    [239, 71, 111],
    [255, 209, 102],
    [6, 214, 160],
    [17, 138, 178],
    [7, 59, 76],
    [131, 56, 236],
], dtype=np.uint8)

def list_images(folder, limit=None):
    # Tra ve danh sach anh trong mot folder theo thu tu alphabet de ket qua on dinh.
    folder = Path(folder)
    if not folder.exists():
        return []
    images = [p for p in sorted(folder.iterdir()) if p.suffix.lower() in IMG_EXTS]
    return images if limit is None else images[:limit]

def count_images(folder):
    # Dem so anh de in nhanh thong tin dataset trong preflight check.
    return len(list_images(folder, limit=None))

def read_image(path):
    # Luon doc anh ve RGB de tranh sai khac so kenh khi visualize.
    return Image.open(path).convert('RGB')

def color_mask(mask):
    # To mau semantic mask bang bang mau co dinh de de nhin tren video.
    mask = np.asarray(mask, dtype=np.int64)
    return PAL[np.clip(mask, 0, len(PAL) - 1)]

def overlay_mask(image, mask, alpha=0.45):
    # Ve semantic mask de len anh goc, chi to mau o pixel co nhan > 0.
    image = np.asarray(image).astype(np.float32)
    mask = np.asarray(mask)
    mask_rgb = color_mask(mask).astype(np.float32)
    valid = (mask > 0).astype(np.float32)[..., None]
    mixed = image * (1 - alpha * valid) + mask_rgb * (alpha * valid)
    return mixed.astype(np.uint8)

def clear_repo_modules():
    # Xoa cache import cua repo truoc do de tranh xung dot giua local-reliability va semantic.
    for name in list(sys.modules):
        if name.split('.')[0] in PACKAGE_PREFIXES:
            del sys.modules[name]

def activate_repo(kind):
    # Dua repo can dung len dau sys.path va chuyen working directory vao repo do.
    clear_repo_modules()
    repo = REPOS[kind]
    sys.path = [p for p in sys.path if Path(p).resolve() != repo.resolve()]
    sys.path.insert(0, str(repo))
    os.chdir(repo)
    return repo

def build_opts(kind):
    # Gop cac override chung de notebook va train/test command tham chieu cung mot nguon.
    spec = APPROACH[kind]
    return [
        'MODEL.DEVICE_ID', '0',
        'MODEL.PRETRAIN_CHOICE', 'imagenet',
        'MODEL.PRETRAIN_PATH', str(PRETRAIN),
        'DATASETS.ROOT_DIR', str(DATA_ROOT),
        'SOLVER.IMS_PER_BATCH', str(spec['ims']),
        'DATALOADER.NUM_WORKERS', str(spec['workers']),
        'SOLVER.MAX_EPOCHS', str(spec['epochs']),
        'SOLVER.EVAL_PERIOD', str(spec['eval_period']),
        'SOLVER.CHECKPOINT_PERIOD', str(spec['ckpt_period']),
        'OUTPUT_DIR', str(spec['output_dir']),
    ]

def print_check(status, message):
    # In theo dinh dang de doc nhanh trong video va trong log notebook.
    print(f'[{status}] {message}')

def preflight_check(require_checkpoint=True):
    # Kiem tra toan bo input can thiet truoc khi chay demo chinh thuc.
    required_dirs = {
        'workspace': WORKSPACE,
        'repo local-reliability': REPOS['local_reliability'],
        'repo semantic': REPOS['semantic'],
        'dataset root': DATASET_ROOT,
        'bounding_box_train': DATASET_ROOT / 'bounding_box_train',
        'query': DATASET_ROOT / 'query',
        'bounding_box_test': DATASET_ROOT / 'bounding_box_test',
        'semantic mask dir': APPROACH['semantic']['mask_dir'],
    }
    missing_items = []
    for label, path in required_dirs.items():
        if Path(path).exists():
            print_check('OK', f'{label}: {path}')
        else:
            print_check('MISSING', f'{label}: {path}')
            missing_items.append(label)

    for subset in ['bounding_box_train', 'query', 'bounding_box_test']:
        folder = DATASET_ROOT / subset
        if folder.exists():
            print_check('OK', f'{subset} image count = {count_images(folder)}')

    if PRETRAIN.exists():
        print_check('OK', f'pretrained backbone: {PRETRAIN}')
    else:
        print_check('MISSING', f'pretrained backbone: {PRETRAIN}')
        missing_items.append('pretrained backbone')

    for kind, spec in APPROACH.items():
        ckpt = spec.get('test_weight')
        if ckpt is None:
            print_check('ERROR', f"{spec['display_name']} chua gan test_weight")
            if require_checkpoint:
                raise FileNotFoundError("Checkpoint is required for official demo. Please set APPROACH[kind]['test_weight'].")
            continue
        ckpt = Path(ckpt)
        if ckpt.exists():
            print_check('OK', f"{spec['display_name']} checkpoint: {ckpt}")
        else:
            print_check('ERROR', f"{spec['display_name']} checkpoint khong ton tai: {ckpt}")
            if require_checkpoint:
                raise FileNotFoundError("Checkpoint is required for official demo. Please set APPROACH[kind]['test_weight'].")

    if missing_items:
        raise FileNotFoundError('Preflight check that bai. Thieu: ' + ', '.join(missing_items))
    print('\nPreflight check thanh cong. Notebook san sang chay evaluation va visualization.')

preflight_check(require_checkpoint=True)
os.chdir(WORKSPACE)


## Dataset Preview, ReID Problem Visualization, va Query-Gallery Setup

Truoc khi chay evaluation, ta can minh hoa ro input cua bai toan Person ReID:
- dataset gom `query` va `gallery` thay vi mot tap classification don gian;
- cung mot nguoi co the xuat hien o nhieu camera khac nhau, goc nhin va dieu kien anh sang khac nhau;
- nhieu nguoi khac nhau co the mac do gan giong nhau, tao distractor kho.

Cell code ben duoi se thuc hien ba viec:
1. preview nhanh ba tap `train`, `query`, `gallery`;
2. hien thi mot identity xuat hien qua nhieu camera;
3. hien thi mot query va mot so gallery candidates de giai thich setup retrieval.


In [ ]:
def parse_reid_filename(path):
    # Tach thong tin pid va camid tu ten file theo kieu Market/Duke, vi du 0001_c1s1_001051_00.jpg.
    name = Path(path).stem
    parts = name.split('_')
    pid = parts[0] if parts else 'unknown'
    cam_match = re.search(r'c(\d+)', name)
    camid = int(cam_match.group(1)) if cam_match else None
    return {'pid': pid, 'camid': camid, 'name': name}

def show_image_row(paths, title, cols=6, figsize_scale=3.5):
    # Hien thi mot hang anh voi title ro rang de de thuyet minh khi quay video.
    paths = list(paths)
    if not paths:
        print(f'Khong co anh de hien thi cho section: {title}')
        return
    cols = min(cols, len(paths))
    fig, axes = plt.subplots(1, cols, figsize=(figsize_scale * cols, figsize_scale))
    axes = np.atleast_1d(axes)
    fig.suptitle(title, fontsize=16)
    for ax, path in zip(axes, paths[:cols]):
        meta = parse_reid_filename(path)
        ax.imshow(read_image(path))
        ax.set_title(f"pid={meta['pid']}\ncam={meta['camid']}", fontsize=11)
        ax.axis('off')
    plt.tight_layout()
    plt.show()

def build_pid_index(folder):
    # Lap chi muc pid -> danh sach anh de tim cung-ID va distractor cho demo.
    index = {}
    for path in list_images(folder, limit=None):
        meta = parse_reid_filename(path)
        index.setdefault(meta['pid'], []).append(path)
    return index

def choose_same_identity_examples(folder, min_images=3, prefer_multi_camera=True, limit=6):
    # Tim mot pid co du nhieu anh, uu tien xuat hien tren nhieu camera de minh hoa cross-camera.
    pid_index = build_pid_index(folder)
    best_paths = []
    for pid, paths in pid_index.items():
        if len(paths) < min_images:
            continue
        cams = {parse_reid_filename(p)['camid'] for p in paths}
        if prefer_multi_camera and len(cams) < 2:
            continue
        best_paths = sorted(paths)[:limit]
        break
    if not best_paths:
        for pid, paths in pid_index.items():
            if len(paths) >= min_images:
                best_paths = sorted(paths)[:limit]
                break
    return best_paths

def choose_distractors(folder, reference_pid=None, limit=6):
    # Lay mot vai nguoi khac pid de lam distractor minh hoa do kho cua bai toan.
    chosen = []
    used_pid = set()
    for path in list_images(folder, limit=None):
        meta = parse_reid_filename(path)
        if meta['pid'] == reference_pid or meta['pid'] in used_pid:
            continue
        chosen.append(path)
        used_pid.add(meta['pid'])
        if len(chosen) >= limit:
            break
    return chosen

def visualize_dataset_preview():
    # Preview nhanh mot vai anh o train, query va gallery de kiem tra du lieu truoc khi demo.
    for subset in ['bounding_box_train', 'query', 'bounding_box_test']:
        paths = list_images(DATASET_ROOT / subset, limit=5)
        show_image_row(paths, title=f'Dataset preview: {subset}', cols=5)

def visualize_reid_problem():
    # Minh hoa cung mot identity qua nhieu camera va distractor khac identity.
    same_id_paths = choose_same_identity_examples(DATASET_ROOT / 'bounding_box_train', min_images=3, prefer_multi_camera=True, limit=6)
    if same_id_paths:
        ref_pid = parse_reid_filename(same_id_paths[0])['pid']
        show_image_row(same_id_paths, title='ReID problem: same identity across cameras', cols=min(6, len(same_id_paths)))
        distractors = choose_distractors(DATASET_ROOT / 'bounding_box_test', reference_pid=ref_pid, limit=6)
        show_image_row(distractors, title='ReID problem: different identities as distractors', cols=min(6, len(distractors)))
    else:
        print('Khong tim duoc vi du same identity phu hop de visualize.')

def visualize_query_gallery_setup():
    # Hien thi mot query va mot so gallery candidates de nhan manh day la bai toan retrieval.
    query_paths = list_images(DATASET_ROOT / 'query', limit=10)
    gallery_paths = list_images(DATASET_ROOT / 'bounding_box_test', limit=12)
    if not query_paths or not gallery_paths:
        print('Khong du anh query/gallery de visualize setup.')
        return
    query_path = query_paths[min(QUERY_INDEX, len(query_paths) - 1)]
    fig, axes = plt.subplots(1, 6, figsize=(22, 4.5))
    qmeta = parse_reid_filename(query_path)
    axes[0].imshow(read_image(query_path))
    axes[0].set_title(f"Query\npid={qmeta['pid']} cam={qmeta['camid']}", fontsize=11)
    axes[0].axis('off')
    for col, path in enumerate(gallery_paths[:5], start=1):
        meta = parse_reid_filename(path)
        axes[col].imshow(read_image(path))
        axes[col].set_title(f"Gallery\npid={meta['pid']} cam={meta['camid']}", fontsize=11)
        axes[col].axis('off')
    fig.suptitle('Query-gallery setup: query la nguoi can tim, gallery la tap can truy hoi', fontsize=16)
    plt.tight_layout()
    plt.show()

visualize_dataset_preview()
visualize_reid_problem()
visualize_query_gallery_setup()
os.chdir(WORKSPACE)


## Semantic Mask Preview va Approach Configuration Summary

Phan nay co hai muc tieu:
- xem nhanh semantic mask da duoc prepare dung cau truc chua;
- in ra `final merged config` o muc tom tat, nhung **khong load model hoac dataloader** de cell nay chay nhanh va gon.

Config summary se cho thay nhung thanh phan quan trong khi demo: backbone, kich thuoc anh, batch size, output dir, checkpoint evaluation va cac co lien quan den reliability / semantic.


In [ ]:
def paired_mask_path(image_path, mask_root):
    # Suy ra duong dan semantic mask tu duong dan anh theo cung ten file va subset.
    image_path = Path(image_path)
    subset = image_path.parent.name
    return Path(mask_root) / subset / f'{image_path.stem}.png'

def preview_semantic_masks(limit=4):
    # Hien thi anh goc, mask va overlay de xac nhan semantic mask da duoc tao dung.
    mask_root = APPROACH['semantic']['mask_dir']
    query_images = list_images(DATASET_ROOT / 'query', limit=None)
    pairs = []
    for img_path in query_images:
        mask_path = paired_mask_path(img_path, mask_root)
        if mask_path.exists():
            pairs.append((img_path, mask_path))
        if len(pairs) >= limit:
            break
    if not pairs:
        print('Khong tim thay cap anh-mask hop le trong semantic_groups.')
        return
    fig, axes = plt.subplots(len(pairs), 3, figsize=(12, 4 * len(pairs)))
    axes = np.atleast_2d(axes)
    for row, (img_path, mask_path) in enumerate(pairs):
        image = read_image(img_path)
        mask = Image.open(mask_path)
        axes[row, 0].imshow(image)
        axes[row, 0].set_title(f'Image\n{img_path.name}', fontsize=11)
        axes[row, 1].imshow(color_mask(mask))
        axes[row, 1].set_title(f'Mask\n{mask_path.name}', fontsize=11)
        axes[row, 2].imshow(overlay_mask(image, mask))
        axes[row, 2].set_title('Overlay', fontsize=11)
        for col in range(3):
            axes[row, col].axis('off')
    fig.suptitle('Semantic mask preview', fontsize=16)
    plt.tight_layout()
    plt.show()

def load_cfg_only(kind):
    # Chi load va merge config. Khong tao dataloader, khong tao model, khong load checkpoint.
    repo = activate_repo(kind)
    from config import cfg as base_cfg
    cfg = base_cfg.clone()
    cfg.merge_from_file(APPROACH[kind]['config'])
    cfg.merge_from_list(build_opts(kind))
    cfg.freeze()
    os.chdir(WORKSPACE)
    return repo, cfg

def summarize_cfg(kind):
    # Tom tat mot so thong tin quan trong cua config de doc nhanh trong phan demo.
    repo, cfg = load_cfg_only(kind)
    sem_enabled = bool(getattr(getattr(cfg.MODEL, 'SEM_ALIGN', object()), 'ENABLED', False))
    summary = {
        'Approach': APPROACH[kind]['display_name'],
        'Repo': str(repo),
        'Config': APPROACH[kind]['config'],
        'Backbone': str(cfg.MODEL.NAME),
        'Dataset': '+'.join(cfg.DATASETS.NAMES),
        'Image size': f"{cfg.INPUT.SIZE_TRAIN[0]}x{cfg.INPUT.SIZE_TRAIN[1]}",
        'Batch size': int(cfg.SOLVER.IMS_PER_BATCH),
        'Output dir': str(cfg.OUTPUT_DIR),
        'Checkpoint': str(APPROACH[kind]['test_weight']),
        'Reliability pipeline': bool(getattr(cfg.MODEL, 'RELIABILITY_PIPELINE', False)),
        'JPM': bool(getattr(cfg.MODEL, 'JPM', False)),
        'Semantic enabled': sem_enabled,
        'Mask dir': str(APPROACH[kind].get('mask_dir', '-')),
    }
    return summary

preview_semantic_masks(limit=3)
cfg_summary_df = pd.DataFrame([summarize_cfg('local_reliability'), summarize_cfg('semantic')])
display(cfg_summary_df)


## Official Retrieval Evaluation

Day la phan evaluation chinh thuc cua notebook. Moi approach se:
- load dataloader evaluation;
- load model va checkpoint tu `test_weight`;
- trich dac trung cho query va gallery;
- tinh khoang cach, `mAP` va `CMC Rank-k`.

Ket qua cuoi cung duoc dua ve mot bang de so sanh hai huong Local Reliability va Semantic Reliability.


In [ ]:
EVAL_CACHE = {}

def load_checkpoint_into_model(model, checkpoint_path):
    # Ho tro ca hai dang checkpoint: state_dict thuan hoac dict co key model/state_dict.
    state = torch.load(checkpoint_path, map_location='cpu')
    if isinstance(state, dict):
        if 'model' in state:
            state = state['model']
        elif 'state_dict' in state:
            state = state['state_dict']
    cleaned = {}
    for key, value in state.items():
        cleaned[key.replace('module.', '')] = value
    model.load_state_dict(cleaned, strict=False)
    return model

def load_eval_context(kind):
    # Tao dataloader validation va model evaluation cho approach can danh gia.
    repo = activate_repo(kind)
    from config import cfg as base_cfg
    from datasets import make_dataloader
    from model import make_model
    cfg = base_cfg.clone()
    cfg.merge_from_file(APPROACH[kind]['config'])
    cfg.merge_from_list(build_opts(kind))
    cfg.freeze()
    train_loader, train_loader_normal, val_loader, num_query, num_classes, camera_num, view_num = make_dataloader(cfg)
    model = make_model(cfg, num_class=num_classes, camera_num=camera_num, view_num=view_num).to(DEVICE)
    model = load_checkpoint_into_model(model, APPROACH[kind]['test_weight'])
    model.eval()
    os.chdir(WORKSPACE)
    return {
        'repo': repo,
        'cfg': cfg,
        'model': model,
        'val_loader': val_loader,
        'num_query': num_query,
    }

def evaluate_retrieval(kind):
    # Chay forward tren toan bo query+gallery, sau do tinh metric bang utility goc cua repo.
    if kind in EVAL_CACHE:
        return EVAL_CACHE[kind]
    ctx = load_eval_context(kind)
    activate_repo(kind)
    from utils.metrics import euclidean_distance, eval_func
    feats, pids, camids, paths = [], [], [], []
    for batch in ctx['val_loader']:
        imgs, pid_batch, camid_list, camids_batch, viewids_batch, path_batch = batch
        with torch.no_grad():
            feat = ctx['model'](imgs.to(DEVICE), cam_label=camids_batch.to(DEVICE), view_label=viewids_batch.to(DEVICE))
        feats.append(feat.cpu())
        pids.extend(list(pid_batch))
        camids.extend(list(camid_list))
        paths.extend(list(path_batch))
    feats = torch.cat(feats, dim=0)
    if ctx['cfg'].TEST.FEAT_NORM == 'yes':
        feats = F.normalize(feats, dim=1)
    num_query = ctx['num_query']
    qf, gf = feats[:num_query], feats[num_query:]
    qp, gp = np.asarray(pids[:num_query]), np.asarray(pids[num_query:])
    qc, gc = np.asarray(camids[:num_query]), np.asarray(camids[num_query:])
    qpaths, gpaths = paths[:num_query], paths[num_query:]
    dist = euclidean_distance(qf, gf)
    cmc, mAP = eval_func(dist, qp, gp, qc, gc)
    result = {
        'kind': kind,
        'display_name': APPROACH[kind]['display_name'],
        'cfg': ctx['cfg'],
        'dist': dist,
        'cmc': cmc,
        'mAP': float(mAP),
        'qp': qp,
        'gp': gp,
        'qc': qc,
        'gc': gc,
        'qpaths': qpaths,
        'gpaths': gpaths,
        'checkpoint': str(APPROACH[kind]['test_weight']),
    }
    EVAL_CACHE[kind] = result
    os.chdir(WORKSPACE)
    return result

def build_metric_table(results):
    # Chuan hoa ket qua evaluation ve mot bang de in dep va de dua vao video.
    rows = []
    for res in results:
        rows.append({
            'Approach': res['display_name'],
            'mAP': round(100 * res['mAP'], 2),
            'Rank-1': round(100 * float(res['cmc'][0]), 2),
            'Rank-5': round(100 * float(res['cmc'][4]), 2) if len(res['cmc']) > 4 else np.nan,
            'Rank-10': round(100 * float(res['cmc'][9]), 2) if len(res['cmc']) > 9 else np.nan,
            'Checkpoint': res['checkpoint'],
        })
    return pd.DataFrame(rows)

evaluation_results = [evaluate_retrieval('local_reliability'), evaluate_retrieval('semantic')]
for res in evaluation_results:
    print(res['display_name'])
    print(f"mAP: {100 * res['mAP']:.2f}")
    print(f"Rank-1: {100 * float(res['cmc'][0]):.2f}")
    print(f"Rank-5: {100 * float(res['cmc'][4]):.2f}")
    print(f"Rank-10: {100 * float(res['cmc'][9]):.2f}\n")

metric_table = build_metric_table(evaluation_results)
display(metric_table)
os.chdir(WORKSPACE)


## Top-k Retrieval Visualization, Successful Case, va Failure Case Analysis

Sau khi da co metric, phan quan trong nhat cua demo la nhin truc quan ket qua truy hoi:
- query o ben trai;
- top-k gallery do model xep hang o ben phai;
- moi anh duoc danh dau `MATCH` hoac `WRONG`.

Cell ben duoi se hien thi:
1. mot top-k retrieval minh hoa cho moi approach;
2. mot successful case voi Top-1 dung;
3. mot failure case voi Top-1 sai neu tim thay.


In [ ]:
def valid_gallery_indices(res, qidx):
    # Loai bo gallery co cung pid va cung camera voi query theo protocol ReID pho bien.
    order = np.argsort(res['dist'][qidx])
    keep = []
    for gi in order:
        if res['gp'][gi] == res['qp'][qidx] and res['gc'][gi] == res['qc'][qidx]:
            continue
        keep.append(gi)
    return keep

def visualize_topk_retrieval(query_path, gallery_paths, query_pid, gallery_pids, gallery_camids, distances=None, topk=10, title=None):
    # Hien thi query va top-k gallery voi thong tin rank, pid, cam va MATCH/WRONG.
    topk = min(topk, len(gallery_paths))
    fig, axes = plt.subplots(1, topk + 1, figsize=(3.2 * (topk + 1), 4.8))
    axes = np.atleast_1d(axes)
    axes[0].imshow(read_image(query_path))
    qmeta = parse_reid_filename(query_path)
    axes[0].set_title(f"Query\npid={query_pid} cam={qmeta['camid']}", fontsize=11)
    axes[0].axis('off')
    for rank in range(topk):
        path = gallery_paths[rank]
        gallery_pid = gallery_pids[rank]
        gallery_cam = gallery_camids[rank]
        is_match = str(gallery_pid) == str(query_pid)
        dist_text = '' if distances is None else f"\ndist={float(distances[rank]):.4f}"
        axes[rank + 1].imshow(read_image(path))
        axes[rank + 1].set_title(
            f"Rank-{rank + 1}\npid={gallery_pid} cam={gallery_cam}{dist_text}\n{'MATCH' if is_match else 'WRONG'}",
            fontsize=10,
        )
        axes[rank + 1].axis('off')
        for spine in axes[rank + 1].spines.values():
            spine.set_visible(True)
            spine.set_linewidth(3)
            spine.set_edgecolor('limegreen' if is_match else 'crimson')
    if title:
        fig.suptitle(title, fontsize=16)
    plt.tight_layout()
    plt.show()

def show_query_result(res, qidx, topk=10, title_prefix='Top-k retrieval'):
    # Lay top-k hop le cho mot query roi dua vao ham visualize.
    keep = valid_gallery_indices(res, qidx)[:topk]
    visualize_topk_retrieval(
        query_path=res['qpaths'][qidx],
        gallery_paths=[res['gpaths'][gi] for gi in keep],
        query_pid=res['qp'][qidx],
        gallery_pids=[res['gp'][gi] for gi in keep],
        gallery_camids=[res['gc'][gi] for gi in keep],
        distances=[res['dist'][qidx][gi] for gi in keep],
        topk=topk,
        title=f"{title_prefix}: {res['display_name']}",
    )

def find_case_index(res, want_success=True, sample_limit=200):
    # Tim query co Top-1 dung hoac sai de tao minh hoa thanh cong / that bai.
    total = min(len(res['qpaths']), sample_limit)
    for qidx in range(total):
        keep = valid_gallery_indices(res, qidx)
        if not keep:
            continue
        top1_match = str(res['gp'][keep[0]]) == str(res['qp'][qidx])
        if top1_match == want_success:
            return qidx
    return None

for res in evaluation_results:
    qidx = min(QUERY_INDEX, len(res['qpaths']) - 1)
    show_query_result(res, qidx=qidx, topk=TOPK, title_prefix='Top-k retrieval visualization')

for res in evaluation_results:
    success_idx = find_case_index(res, want_success=True)
    if success_idx is not None:
        show_query_result(res, qidx=success_idx, topk=min(5, TOPK), title_prefix='Successful retrieval case')
        print(f"{res['display_name']}: Top-1 dung o query index {success_idx}.")
    else:
        print(f"{res['display_name']}: khong tim thay successful case trong mau da quet.")

    failure_idx = find_case_index(res, want_success=False)
    if failure_idx is not None:
        show_query_result(res, qidx=failure_idx, topk=min(5, TOPK), title_prefix='Failure case analysis')
        print('Phan tich nhanh: Top-1 sai thuong den tu trang phuc giong nhau, occlusion, pose khac, anh mo, hoac bias camera/background.')
    else:
        print('No obvious failure case found in sampled queries.')


## Reliability and Semantic Visualization

Phan nay dung de giai thich y tuong mo hinh thay vi chi dua ra con so metric.

Goi y cach dien giai khi quay video:
- reliability heatmap cho thay mo hinh danh gia muc do dang tin cua tung patch;
- cac vung chua thong tin dinh danh nhu ao, quan, ba lo, giay thuong duoc trong so cao hon;
- background, vung bi che, hoac patch kem thong tin se co do tin cay thap hon;
- voi semantic reliability, semantic mask va semantic prediction giup mo hinh dinh vi cac vung co nghia tren co the nguoi.


In [ ]:
VIS_CONTEXT_CACHE = {}

def load_visual_context(kind):
    # Nap model va dataloader mot lan de dung cho visualization noi bo.
    if kind in VIS_CONTEXT_CACHE:
        return VIS_CONTEXT_CACHE[kind]
    repo = activate_repo(kind)
    from config import cfg as base_cfg
    from datasets import make_dataloader
    from model import make_model
    cfg = base_cfg.clone()
    cfg.merge_from_file(APPROACH[kind]['config'])
    cfg.merge_from_list(build_opts(kind))
    cfg.freeze()
    train_loader, train_loader_normal, val_loader, num_query, num_classes, camera_num, view_num = make_dataloader(cfg)
    model = make_model(cfg, num_class=num_classes, camera_num=camera_num, view_num=view_num).to(DEVICE)
    model = load_checkpoint_into_model(model, APPROACH[kind]['test_weight'])
    model.eval()
    ctx = {
        'repo': repo,
        'cfg': cfg,
        'model': model,
        'train_loader': train_loader,
        'val_loader': val_loader,
    }
    VIS_CONTEXT_CACHE[kind] = ctx
    os.chdir(WORKSPACE)
    return ctx

def denorm_image(img_tensor, cfg):
    # Dua tensor da normalize ve RGB [0, 1] de ve lai dung mau.
    mean = torch.tensor(cfg.INPUT.PIXEL_MEAN, dtype=img_tensor.dtype).view(-1, 1, 1)
    std = torch.tensor(cfg.INPUT.PIXEL_STD, dtype=img_tensor.dtype).view(-1, 1, 1)
    img = (img_tensor.cpu() * std + mean).clamp(0, 1)
    return img.permute(1, 2, 0).numpy()

def tensor_to_heatmap(values, patch_grid, output_size):
    # Noi suy vector patch ve kich thuoc anh de tao heatmap de doc tren video.
    values = torch.as_tensor(values).float().reshape(1, 1, patch_grid[0], patch_grid[1])
    heat = F.interpolate(values, size=output_size, mode='bilinear', align_corners=False)[0, 0]
    return heat.cpu().numpy()

def show_heatmap(ax, image, heatmap, title, cmap='magma'):
    # Ve anh goc va phu them heatmap de nguoi xem thay duoc vi tri mo hinh chu y.
    ax.imshow(image)
    if heatmap is not None:
        ax.imshow(heatmap, cmap=cmap, alpha=0.55)
    ax.set_title(title, fontsize=11)
    ax.axis('off')

def first_batch(loader):
    # Lay nhanh mot batch dau tien de demo interpretability.
    return next(iter(loader))

def visualize_local_reliability_demo():
    # Local-reliability khong tra feature_dict o eval, vi vay ta lay aux_outputs bang train-mode va no_grad.
    ctx = load_visual_context('local_reliability')
    batch = first_batch(ctx['train_loader'])
    imgs, pids, camids, viewids = batch
    img = imgs[:1].to(DEVICE)
    pid = pids[:1].to(DEVICE)
    cam = camids[:1].to(DEVICE)
    view = viewids[:1].to(DEVICE)
    was_training = ctx['model'].training
    ctx['model'].train()
    with torch.no_grad():
        outputs = ctx['model'](img, label=pid, cam_label=cam, view_label=view)
    if not was_training:
        ctx['model'].eval()
    aux = outputs[2] if isinstance(outputs, tuple) and len(outputs) == 3 else outputs[-1]
    image = denorm_image(imgs[0], ctx['cfg'])
    patch_grid = aux.get('patch_grid', (21, 10))
    patch_weights = aux.get('patch_weights')
    patch_reliability = aux.get('patch_reliability')
    selection_mask = aux.get('selection_mask')
    if selection_mask is None:
        selection_mask = aux.get('foreground_keep_mask')
    local_weights = aux.get('local_weights')

    def squeeze_map(x):
        # Chuan hoa tensor ve vector 1D cua mot sample de ve heatmap.
        if x is None:
            return None
        x = torch.as_tensor(x).detach().cpu()
        while x.ndim > 1:
            x = x[0]
        return x.numpy()

    weight_map = tensor_to_heatmap(squeeze_map(patch_weights), patch_grid, image.shape[:2]) if patch_weights is not None else None
    reliability_map = tensor_to_heatmap(squeeze_map(patch_reliability), patch_grid, image.shape[:2]) if patch_reliability is not None else None
    selection_map = tensor_to_heatmap(squeeze_map(selection_mask), patch_grid, image.shape[:2]) if selection_mask is not None else None

    fig, axes = plt.subplots(1, 4, figsize=(22, 5))
    axes[0].imshow(image)
    axes[0].set_title('Original image', fontsize=11)
    axes[0].axis('off')
    show_heatmap(axes[1], image, weight_map, 'Patch weights')
    show_heatmap(axes[2], image, reliability_map, 'Patch reliability')
    show_heatmap(axes[3], image, selection_map, 'Selection / keep mask')
    fig.suptitle('Local Reliability interpretability view', fontsize=16)
    plt.tight_layout()
    plt.show()

    if local_weights is not None:
        local_weights = torch.as_tensor(local_weights)[0].detach().cpu().numpy()
        plt.figure(figsize=(8, 4))
        plt.bar(np.arange(1, len(local_weights) + 1), local_weights, color='teal')
        plt.title('JPM local branch weights')
        plt.xlabel('Local branch')
        plt.ylabel('Weight')
        plt.show()
    else:
        warnings.warn('Local Reliability: khong co key local_weights de ve bar chart.')

def visualize_semantic_reliability_demo():
    # Semantic repo cho phep lay feature_dict, nhờ đó co the xem patch weights va semantic prediction truc tiep.
    ctx = load_visual_context('semantic')
    batch = first_batch(ctx['train_loader'])
    if len(batch) == 5:
        imgs, pids, camids, viewids, semantic_masks = batch
    else:
        raise RuntimeError('Semantic train loader khong tra semantic mask. Hay kiem tra semantic_groups va config.')
    img = imgs[:1].to(DEVICE)
    cam = camids[:1].to(DEVICE)
    view = viewids[:1].to(DEVICE)
    mask = semantic_masks[:1].to(DEVICE)
    with torch.no_grad():
        feature_dict = ctx['model'](img, cam_label=cam, view_label=view, return_feature_dict=True, semantic_masks=mask)
    image = denorm_image(imgs[0], ctx['cfg'])
    gt_mask = semantic_masks[0].detach().cpu().numpy()
    pred_logits = feature_dict.get('semantic_pixel_logits')
    pred_mask = None
    if pred_logits is not None:
        pred_mask = torch.argmax(pred_logits[0], dim=0).detach().cpu().numpy()
    patch_grid = feature_dict.get('patch_grid', (21, 10))
    patch_weights = feature_dict.get('patch_weights')
    patch_reliability = feature_dict.get('patch_reliability')
    selection_mask = feature_dict.get('selection_mask')

    def squeeze_map(x):
        # Chuyen tensor dau ra ve vector 1D cho mot anh de noi suy len heatmap.
        if x is None:
            return None
        x = torch.as_tensor(x).detach().cpu()
        while x.ndim > 1:
            x = x[0]
        return x.numpy()

    weight_map = tensor_to_heatmap(squeeze_map(patch_weights), patch_grid, image.shape[:2]) if patch_weights is not None else None
    reliability_map = tensor_to_heatmap(squeeze_map(patch_reliability), patch_grid, image.shape[:2]) if patch_reliability is not None else None
    selection_map = tensor_to_heatmap(squeeze_map(selection_mask), patch_grid, image.shape[:2]) if selection_mask is not None else None

    fig, axes = plt.subplots(2, 3, figsize=(16, 10))
    axes[0, 0].imshow(image)
    axes[0, 0].set_title('Original image', fontsize=11)
    axes[0, 0].axis('off')
    axes[0, 1].imshow(color_mask(gt_mask))
    axes[0, 1].set_title('GT semantic mask', fontsize=11)
    axes[0, 1].axis('off')
    axes[0, 2].imshow(overlay_mask((image * 255).astype(np.uint8), gt_mask))
    axes[0, 2].set_title('GT overlay', fontsize=11)
    axes[0, 2].axis('off')
    if pred_mask is not None:
        axes[1, 0].imshow(color_mask(pred_mask))
        axes[1, 0].set_title('Predicted semantic map', fontsize=11)
        axes[1, 0].axis('off')
    else:
        axes[1, 0].text(0.5, 0.5, 'Khong co\nsemantic prediction', ha='center', va='center')
        axes[1, 0].axis('off')
        warnings.warn('Semantic Reliability: khong co key semantic_pixel_logits.')
    show_heatmap(axes[1, 1], image, reliability_map, 'Patch reliability')
    show_heatmap(axes[1, 2], image, selection_map if selection_map is not None else weight_map, 'Selection mask / semantic-aware weights')
    fig.suptitle('Semantic Reliability interpretability view', fontsize=16)
    plt.tight_layout()
    plt.show()

    if weight_map is None:
        warnings.warn('Semantic Reliability: khong co key patch_weights de ve heatmap.')

visualize_local_reliability_demo()
visualize_semantic_reliability_demo()
os.chdir(WORKSPACE)


## Summary for Video Presentation

Co the dung cac y chinh duoi day de ket video:

- Notebook nay demo chinh thuc Person ReID sau khi da co checkpoint hop le.
- Person ReID la bai toan **query-gallery retrieval**, khong phai classification don thuan.
- Visualization same-ID cross-camera cho thay cung mot nguoi co the thay doi manh theo camera, pose, anh sang va muc do che khuat.
- Evaluation dung `mAP` va `CMC Rank-k`: `Rank-1` cho biet nguoi dung co dung o vi tri dau khong, con `mAP` phan anh chat luong xep hang tong the.
- Top-k retrieval visualization giup kiem tra truc quan mo hinh truy hoi dung hay sai trong tung query cu the.
- Reliability heatmap va semantic mask giup giai thich vung nao tren anh duoc mo hinh xem la dang tin de nhan dang danh tinh.

Neu can rut gon khi quay video, co the chay theo thu tu: preflight -> ReID problem -> official evaluation -> top-k retrieval -> reliability/semantic visualization -> ket luan.


In [ ]:
def build_train_test_commands(kind):
    # In lai lenh train/test tham khao de notebook van tu tai lieu day du, du chinh demo tap trung vao checkpoint.
    repo = REPOS[kind]
    spec = APPROACH[kind]
    train_cmd = ['python', 'train.py', '--config_file', spec['config']] + build_opts(kind)
    test_opts = build_opts(kind) + ['TEST.WEIGHT', str(spec['test_weight'])]
    test_cmd = ['python', 'test.py', '--config_file', spec['config']] + test_opts
    return {
        'repo': repo,
        'train': ' '.join(shlex.quote(str(x)) for x in train_cmd),
        'test': ' '.join(shlex.quote(str(x)) for x in test_cmd),
    }

for kind in ['local_reliability', 'semantic']:
    cmds = build_train_test_commands(kind)
    print(f"\n=== {APPROACH[kind]['display_name']} ===")
    print('Repo :', cmds['repo'])
    print('Train:', cmds['train'])
    print('Test :', cmds['test'])
